# 🤖 Notebook 3: mBERT Fine-tuning

**Transformer-based NER using Multilingual BERT**

This notebook:
1. Loads the Hindi NER dataset
2. Initializes mBERT (`bert-base-multilingual-cased`)
3. Handles subword tokenization and label alignment
4. Fine-tunes mBERT for token classification
5. Evaluates with entity-level P/R/F1
6. Saves the model, predictions, and training curves

⚠️ **GPU recommended.** This notebook will work on CPU but will be slow (~hours vs ~minutes on GPU).

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import json
import warnings
warnings.filterwarnings('ignore')

from src.data_utils import load_hiner_dataset, ID_TO_LABEL, LABEL_LIST
from src.transformer_ner import HindiTransformerNER
from src.evaluation import save_results, plot_training_curves

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Dataset

In [ ]:
dataset = load_hiner_dataset()

train_dataset = dataset['train']
val_key = 'validation' if 'validation' in dataset else 'dev'
val_dataset = dataset[val_key]
test_dataset = dataset['test']

print(f"Train: {len(train_dataset)} examples")
print(f"Val:   {len(val_dataset)} examples")
print(f"Test:  {len(test_dataset)} examples")

## 2. Initialize mBERT

In [ ]:
model = HindiTransformerNER(
    model_name='bert-base-multilingual-cased',
    max_length=128,
    seed=42,
)

## 3. Understand Subword Tokenization

Before fine-tuning, let's see how mBERT tokenizes Hindi text.
This is **the key technical detail** for your presentation.

In [ ]:
# Demonstrate subword tokenization on Hindi
example_sentences = [
    ["नरेंद्र", "मोदी", "ने", "नई", "दिल्ली", "में", "भाषण", "दिया"],
    ["माइक्रोसॉफ्ट", "के", "अधिकारियों", "से", "मुलाकात", "की"],
]

print("Subword Tokenization Examples:")
print("=" * 60)
for sent in example_sentences:
    print(f"\nOriginal: {' '.join(sent)}")
    tokenized = model.tokenizer(
        sent, is_split_into_words=True, 
        return_tensors=None, add_special_tokens=True
    )
    subwords = model.tokenizer.convert_ids_to_tokens(tokenized['input_ids'])
    word_ids = tokenized.word_ids()
    
    print(f"{'Subword':20s} {'Word ID':10s}")
    print("-" * 35)
    for sw, wid in zip(subwords, word_ids):
        wid_str = str(wid) if wid is not None else 'special'
        original = sent[wid] if wid is not None else '-'
        print(f"{sw:20s} {wid_str:10s} ← {original}")

## 4. Fine-tune mBERT

This will take:
- ~1–2 hours on GPU (T4/V100)
- ~6–12 hours on CPU

If on CPU, reduce `num_epochs` to 3–5 for a quick test.

In [ ]:
# Fine-tune
train_results = model.train(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    output_dir='../models/mbert',
    learning_rate=2e-5,
    batch_size=16,           # Reduce to 8 if GPU OOM
    num_epochs=10,           # Reduce to 3-5 if on CPU
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,               # Auto-disabled on CPU
)

## 5. Training Curves

In [ ]:
# Plot training curves
plot_training_curves(
    train_results['log_history'],
    save_path='../results/figures/mbert_training_curves.png'
)

## 6. Evaluate on Test Set

In [ ]:
# Evaluate on test set
test_metrics = model.evaluate(test_dataset)

print("\n" + "=" * 60)
print("mBERT — TEST SET RESULTS")
print("=" * 60)
print(f"\nOverall Precision: {test_metrics['overall_precision']:.4f}")
print(f"Overall Recall:    {test_metrics['overall_recall']:.4f}")
print(f"Overall F1:        {test_metrics['overall_f1']:.4f}")
print(f"\nDetailed Report:\n")
print(test_metrics['report_str'])

## 7. Save Predictions

In [ ]:
# Save metrics
save_results(test_metrics, '../results/mbert_test_metrics.json')

# Save predictions for error analysis
mbert_predictions = model.get_predictions_with_tokens(test_dataset)

with open('../results/predictions/mbert_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(mbert_predictions, f, ensure_ascii=False, indent=2)
print("✓ mBERT predictions saved")

# Save training log
with open('../results/mbert_training_log.json', 'w') as f:
    json.dump(train_results['log_history'], f, indent=2)
print("✓ Training log saved")

## 8. Interactive Demo

In [ ]:
# Demo: predict on custom Hindi sentences
demo_sentences = [
    "नरेंद्र मोदी ने नई दिल्ली में माइक्रोसॉफ्ट के अधिकारियों से मुलाकात की",
    "विराट कोहली मुंबई में एक कार्यक्रम में शामिल हुए",
    "राहुल ने Google में नौकरी शुरू की",
]

print("\n" + "=" * 60)
print("INTERACTIVE DEMO — mBERT Hindi NER")
print("=" * 60)

for sentence in demo_sentences:
    print(f"\n📝 Input: {sentence}")
    results = model.predict_sentence(sentence)
    print(f"{'Token':20s} {'Predicted Label':15s}")
    print("-" * 40)
    for r in results:
        marker = '→ ' if r['label'] != 'O' else '  '
        print(f"{marker}{r['token']:20s} {r['label']:15s}")